In [34]:
import pandas as pd

base = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"
fs = pd.read_csv(base + "finess_sygen_final.csv", dtype=str)
insee = pd.read_csv(base + "insee_communes_final.csv", dtype=str)

In [35]:
def harmoniser(code):
    if pd.isna(code):
        return code
    if "75101" <= code <= "75120":
        return "75056"   # Paris
    if "69381" <= code <= "69389":
        return "69123"   # Lyon
    if "13201" <= code <= "13216":
        return "13055"   # Marseille
    return code

fs2 = fs.dropna(subset=["CODGEO"]).copy()
fs2["CODGEO_ORIG"] = fs2["CODGEO"]
fs2["CODGEO"] = fs2["CODGEO"].map(harmoniser)
fs3 = fs2[fs2["ETAT"] == "A"].copy()

print(len(fs), len(fs2), len(fs3))

3988 3940 3921


In [36]:
cols_num = ["LIT_MED", "LIT_CHI", "LIT_OBS", "LIT_MCO", "LIT_SMR", "LIT_USLD",
            "NB_SCAN", "NB_IRM", "NB_MAMM"]
for c in cols_num:
    fs3[c] = pd.to_numeric(fs3[c], errors="coerce")

hopitaux = fs3.groupby("CODGEO")[cols_num].sum(min_count=1)
hopitaux["nb_etablissements"] = fs3.groupby("CODGEO")["FI"].count()
hopitaux = hopitaux.reset_index()

print(hopitaux.shape)
print(hopitaux["nb_etablissements"].sum())
print(hopitaux.head())

(1714, 11)
3921
  CODGEO  LIT_MED  LIT_CHI  LIT_OBS  LIT_MCO  LIT_SMR  LIT_USLD  NB_SCAN  \
0  01004     31.0     27.0     15.0     73.0      NaN       NaN      NaN   
1  01033      NaN      NaN      NaN      NaN      NaN       NaN      NaN   
2  01034     56.0     19.0      8.0     83.0     40.0       NaN      1.0   
3  01053     26.0     44.0      NaN     70.0    156.0      11.0      0.0   
4  01082      NaN      NaN      NaN      NaN     70.0       NaN      NaN   

   NB_IRM  NB_MAMM  nb_etablissements  
0     NaN      NaN                  3  
1     NaN      NaN                  1  
2     0.0      1.0                  2  
3     0.0      0.0                  5  
4     NaN      NaN                  1  


## Partie 2 : fusion avec les données INSEE

In [37]:
# communes avec établissement absentes de l'INSEE 
print(sorted(set(hopitaux["CODGEO"]) - set(insee["CODGEO"])))

fusion = insee.merge(hopitaux, on="CODGEO", how="left", validate="one_to_one")
fusion["nb_etablissements"] = fusion["nb_etablissements"].fillna(0)

print(fusion.shape)
print("communes avec au moins 1 établissement :", (fusion["nb_etablissements"] > 0).sum())
print(fusion[fusion["CODGEO"] == "01004"].T)

['97123', '97127', '97502', '97606', '97611', '97615']
(34858, 45)
communes avec au moins 1 établissement : 1708
                                                2
CODGEO                                      01004
LIBGEO                          Ambérieu-en-Bugey
SEXE1_AGEPYR1000                 291.774098436508
SEXE1_AGEPYR1003                 327.193219345162
SEXE1_AGEPYR1006                 525.370991705619
SEXE1_AGEPYR1011                 719.637561687717
SEXE1_AGEPYR1018                 733.735171978173
SEXE1_AGEPYR1025                 1581.24638491537
SEXE1_AGEPYR1040                 1344.79206887329
SEXE1_AGEPYR1055                 833.964092706198
SEXE1_AGEPYR1065                 888.674859386373
SEXE1_AGEPYR1080                 244.547759254982
SEXE2_AGEPYR1000                 323.969423155764
SEXE2_AGEPYR1003                 283.369968380778
SEXE2_AGEPYR1006                  467.59143190407
SEXE2_AGEPYR1011                 664.536778087955
SEXE2_AGEPYR1018                  736

In [38]:
print("établissements dans la fusion :", fusion["nb_etablissements"].sum(), "sur", len(fs3))
print("codes 97 dans l'INSEE :", insee["CODGEO"].str.startswith("97").sum())
print(fs3[fs3["CODGEO"].isin(['97123', '97127', '97502', '97606', '97611', '97615'])][["CODGEO", "NOM", "CATEGORIE"]])


établissements dans la fusion : 3906.0 sur 3921
codes 97 dans l'INSEE : 112
     CODGEO                                                NOM CATEGORIE
3853  97123  CENTRE HOSPITALIER IRENEE DE BRUYN EX HÔPITAL ...       106
3855  97127          CENTRE HOSPITALIER LOUIS CONSTANT FLEMING       355
3868  97127                      ANTENNE AUDRA DE SAINT MARTIN       146
3875  97127                                  HAD  ILES DU NORD       127
3879  97123  CH LOUIS CONSTANT FLEMING - ANTENNE SMUR DE SA...       355
3880  97123            CLINIQUE DE CHOISY HAD SAINT BARTHELEMY       127
3881  97127                                    CLINIQUE WATAKI       109
3980  97502                 CENTRE HOSPITALIER  FRANCOIS DUNAN       355
3981  97611                      CENTRE HOSPITALIER DE MAYOTTE       355
3982  97611               UNITE AUTODIALYSE ASSISTEE MAMOUDZOU       141
3983  97611                                         UDM KAWENI       141
3984  97606                   UNITE D'AUTODIALYS

In [39]:
outre_mer = insee[insee["CODGEO"].str.startswith("97")]
print(len(outre_mer))
print(outre_mer["CODGEO"].str[:3].value_counts().sort_index())

112
CODGEO
971    32
972    34
973    22
974    24
Name: count, dtype: int64


**Périmètre.** 15 établissements actifs (0,4 % du total), situés à Saint-Martin, Saint-Barthélemy, Saint-Pierre-et-Miquelon et Mayotte, sont exclus : ces territoires sont absents des tables INSEE de population, de revenus et de densité, ce qui empêche de calculer un indicateur pour eux. L'analyse couvre la France métropolitaine et les quatre départements d'outre-mer historiques (Guadeloupe, Martinique, Guyane, La Réunion)

In [40]:
hors_insee = hopitaux[~hopitaux["CODGEO"].isin(insee["CODGEO"])]
print(len(hors_insee), "communes hors table INSEE,",
      int(hors_insee["nb_etablissements"].sum()), "établissements exclus")

6 communes hors table INSEE, 15 établissements exclus


In [41]:
fusion["nb_etablissements"] = fusion["nb_etablissements"].fillna(0).astype(int)

In [42]:
print(fusion["nb_etablissements"].dtype)
print(fusion["nb_etablissements"].isna().sum())
print((fusion["nb_etablissements"] == 0).sum())

int64
0
33150


In [43]:
fusion.head()

,CODGEO,LIBGEO,SEXE1_AGEPYR1000,SEXE1_AGEPYR1003,SEXE1_AGEPYR1006,SEXE1_AGEPYR1011,SEXE1_AGEPYR1018,SEXE1_AGEPYR1025,SEXE1_AGEPYR1040,SEXE1_AGEPYR1055,...,LIT_MED,LIT_CHI,LIT_OBS,LIT_MCO,LIT_SMR,LIT_USLD,NB_SCAN,NB_IRM,NB_MAMM,nb_etablissements
0,01001,L'Abergement-Clémenciat,19.0,15.0,36.0,42.0,15.0,77.0,87.0,66.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,01002,L'Abergement-de-Varey,3.0,6.0,7.0,22.0,6.0,20.0,39.0,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,01004,Ambérieu-en-Bugey,291.774098436508,327.193219345162,525.370991705619,719.637561687717,733.735171978173,1581.24638491537,1344.79206887329,833.964092706198,...,31.0,27.0,15.0,73.0,NaN,NaN,NaN,NaN,NaN,3
3,01005,Ambérieux-en-Dombes,36.2624041600494,37.108922388434,65.9432761131205,80.0820466018419,54.4457784678517,193.448597496766,196.440979404838,137.738813251787,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,01006,Ambléon,2.0,1.0,2.0,1.0,4.0,12.0,11.0,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [44]:
import os

chemin = "s3://emma457/Projet_python/fusion_insee_hopitaux.csv"
opts = {"client_kwargs": {"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]}}

fusion.to_csv(chemin, index=False, storage_options=opts)
print("Fichier sauvegardé :", chemin)

Fichier sauvegardé : s3://emma457/Projet_python/fusion_insee_hopitaux.csv


Etudes par densité des communautés

In [45]:
dens = pd.read_csv(base + "densite_communes_2025.csv", dtype=str)
cols_dens = ["DENS", "LIBDENS", "DENS7", "LIBDENS7"]

fusion = fusion.drop(columns=cols_dens).merge(
    dens[["CODGEO"] + cols_dens], on="CODGEO", how="left", validate="one_to_one"
)
print(fusion["LIBDENS"].isna().sum(), fusion.shape)   # attendu : 0 (34858, 23)

0 (34858, 45)


Regardons par population

In [46]:
#fusion["a_etablissement"] = fusion["nb_etablissements"] > 0
#fusion["population"] = pd.to_numeric(fusion["population"], errors="coerce")

#tableau = fusion.groupby("LIBDENS", dropna=False)["a_etablissement"].agg(
    #nb_communes="count", part_communes_equipees="mean"
)
#tableau["part_communes_equipees"] = (tableau["part_communes_equipees"] * 100).round(1)

#pop_tot = fusion.groupby("LIBDENS", dropna=False)["population"].sum()
#pop_avec = fusion[fusion["a_etablissement"]].groupby("LIBDENS", dropna=False)["population"].sum()
#tableau["part_population_desservie"] = (pop_avec / pop_tot * 100).round(1)

#print(tableau)

SyntaxError: unmatched ')' (1315846186.py, line 6)

## Partie 3 : fusion avec l'offre de soins de ville (RPPS)

In [ ]:
path_insee = "s3://mangane/Projet_PDS_2026/processed/insee_communes_final.csv"
path_rpps = "s3://mangane/Projet_PDS_2026/processed/offre_soins_commune.csv"
path_finess_sygen = "s3://mangane/Projet_PDS_2026/processed/finess_sygen_final.csv"

insee = pd.read_csv(
    path_insee,
    dtype={"CODGEO": str},
    low_memory=False
)

rpps = pd.read_csv(
    path_rpps,
    dtype={"Code commune (coord. structure)": str},
    low_memory=False
)

finess_sygen = pd.read_csv(
    path_finess_sygen,
    dtype={
        "CODGEO": str,
        "FI": str,
        "FI_EJ": str
    },
    low_memory=False
)

print("INSEE :", insee.shape)
print("RPPS :", rpps.shape)
print("FINESS-SYGEN :", finess_sygen.shape)

INSEE : (34858, 35)
RPPS : (17213, 3)
FINESS-SYGEN : (3988, 42)


In [ ]:
rpps = rpps.rename(
    columns={
        "Code commune (coord. structure)": "CODGEO",
        "Libellé commune (coord. structure)": "LIBGEO_RPPS"
    }
)

insee["CODGEO"] = insee["CODGEO"].astype("string")
rpps["CODGEO"] = rpps["CODGEO"].astype("string")
finess_sygen["CODGEO"] = finess_sygen["CODGEO"].astype("string")

rpps.head()

,CODGEO,LIBGEO_RPPS,nb_professionnels
0,00000,Inconnu,9
1,01001,L'Abergement-Clémenciat,1
2,01002,L'Abergement-de-Varey,1
3,01004,Ambérieu-en-Bugey,521
4,01005,Ambérieux-en-Dombes,29


In [ ]:
print("INSEE")
print("Communes distinctes :", insee["CODGEO"].nunique())
print("CODGEO manquants :", insee["CODGEO"].isna().sum())
print("Doublons CODGEO :", insee["CODGEO"].duplicated().sum())

print("\nRPPS")
print("Communes distinctes :", rpps["CODGEO"].nunique())
print("CODGEO manquants :", rpps["CODGEO"].isna().sum())
print("Doublons CODGEO :", rpps["CODGEO"].duplicated().sum())

print("\nFINESS-SYGEN")
print("Communes distinctes :", finess_sygen["CODGEO"].nunique())
print("CODGEO manquants :", finess_sygen["CODGEO"].isna().sum())
print("Établissements distincts :", finess_sygen["FI"].nunique())

assert not insee["CODGEO"].duplicated().any()
assert not rpps["CODGEO"].duplicated().any()

INSEE
Communes distinctes : 34858
CODGEO manquants : 0
Doublons CODGEO : 0

RPPS
Communes distinctes : 17213
CODGEO manquants : 0
Doublons CODGEO : 0

FINESS-SYGEN
Communes distinctes : 1757
CODGEO manquants : 48
Établissements distincts : 3988


La table FINESS-SYGEN contient une ligne par établissement.

On calcule :
- le nombre d'établissements distincts ;
- les capacités en lits et places ;
- les effectifs ;
- les principaux équipements.

Pour les variables quantitatives, on utilise `sum(min_count=1)` afin de conserver `NaN` lorsqu'aucune valeur n'est renseignée.

In [ ]:
variables_somme = [
    "LIT_MED",
    "LIT_CHI",
    "LIT_OBS",
    "LIT_MCO",
    "PLA_MED",
    "PLA_CHI",
    "PLA_OBS",
    "PLA_MCO",
    "CAP_HC_TOT",
    "CAP_HDJ_TOT",
    "LIT_SMR",
    "PLA_SMR",
    "LIT_USLD",
    "PLATOT_HAD",
    "EFFSAL_TOT",
    "EFFLIB_TOT",
    "ETPSAL_TOT",
    "EFFPNM_TOT_PNM",
    "ETPPNM_TOT_PNM",
    "NB_SCAN",
    "NB_IRM",
    "NB_MAMM"
]

for col in variables_somme:
    finess_sygen[col] = pd.to_numeric(
        finess_sygen[col],
        errors="coerce"
    )

In [ ]:
finess_commune = (
    finess_sygen
    .dropna(subset=["CODGEO"])
    .groupby("CODGEO")
    .agg(
        nb_etablissements=("FI", "nunique"),
        **{
            col: (col, lambda x: x.sum(min_count=1))
            for col in variables_somme
        }
    )
    .reset_index()
)

print("Dimensions FINESS-SYGEN agrégé :", finess_commune.shape)
print("Communes distinctes :", finess_commune["CODGEO"].nunique())

finess_commune.head()

Dimensions FINESS-SYGEN agrégé : (1757, 24)
Communes distinctes : 1757


,CODGEO,nb_etablissements,LIT_MED,LIT_CHI,LIT_OBS,LIT_MCO,PLA_MED,PLA_CHI,PLA_OBS,PLA_MCO,...,LIT_USLD,PLATOT_HAD,EFFSAL_TOT,EFFLIB_TOT,ETPSAL_TOT,EFFPNM_TOT_PNM,ETPPNM_TOT_PNM,NB_SCAN,NB_IRM,NB_MAMM
0,01004,3,31.0,27.0,15.0,73.0,8.0,48.0,0.0,56.0,...,NaN,70.0,17.0,52.0,9.21,300.0,253.73,NaN,NaN,NaN
1,01033,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,4.0,1.01,79.0,74.21,NaN,NaN,NaN
2,01034,2,56.0,19.0,8.0,83.0,8.0,8.0,1.0,17.0,...,NaN,NaN,46.0,1.0,27.88,453.0,395.08,1.0,0.0,1.0
3,01053,5,26.0,44.0,NaN,70.0,5.0,53.0,NaN,58.0,...,11.0,NaN,74.0,82.0,57.76,1574.0,1378.82,0.0,0.0,0.0
4,01082,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5.0,0.0,4.39,107.0,94.63,NaN,NaN,NaN


In [ ]:
finess_commune = finess_commune.rename(
    columns={
        "LIT_MED": "nb_lits_med",
        "LIT_CHI": "nb_lits_chi",
        "LIT_OBS": "nb_lits_obs",
        "LIT_MCO": "nb_lits_mco",
        "PLA_MED": "nb_places_med",
        "PLA_CHI": "nb_places_chi",
        "PLA_OBS": "nb_places_obs",
        "PLA_MCO": "nb_places_mco",
        "CAP_HC_TOT": "capacite_hospitalisation_complete",
        "CAP_HDJ_TOT": "capacite_hospitalisation_jour",
        "LIT_SMR": "nb_lits_smr",
        "PLA_SMR": "nb_places_smr",
        "LIT_USLD": "nb_lits_usld",
        "PLATOT_HAD": "nb_places_had",
        "EFFSAL_TOT": "effectif_salarie_total",
        "EFFLIB_TOT": "effectif_liberal_total",
        "ETPSAL_TOT": "etp_salarie_total",
        "EFFPNM_TOT_PNM": "effectif_personnel_non_medical",
        "ETPPNM_TOT_PNM": "etp_personnel_non_medical",
        "NB_SCAN": "nb_scanners",
        "NB_IRM": "nb_irm",
        "NB_MAMM": "nb_mammographes"
    }
)

finess_commune.head()

,CODGEO,nb_etablissements,nb_lits_med,nb_lits_chi,nb_lits_obs,nb_lits_mco,nb_places_med,nb_places_chi,nb_places_obs,nb_places_mco,...,nb_lits_usld,nb_places_had,effectif_salarie_total,effectif_liberal_total,etp_salarie_total,effectif_personnel_non_medical,etp_personnel_non_medical,nb_scanners,nb_irm,nb_mammographes
0,01004,3,31.0,27.0,15.0,73.0,8.0,48.0,0.0,56.0,...,NaN,70.0,17.0,52.0,9.21,300.0,253.73,NaN,NaN,NaN
1,01033,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,4.0,1.01,79.0,74.21,NaN,NaN,NaN
2,01034,2,56.0,19.0,8.0,83.0,8.0,8.0,1.0,17.0,...,NaN,NaN,46.0,1.0,27.88,453.0,395.08,1.0,0.0,1.0
3,01053,5,26.0,44.0,NaN,70.0,5.0,53.0,NaN,58.0,...,11.0,NaN,74.0,82.0,57.76,1574.0,1378.82,0.0,0.0,0.0
4,01082,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5.0,0.0,4.39,107.0,94.63,NaN,NaN,NaN


In [ ]:
codes_insee = set(insee["CODGEO"].dropna())
codes_rpps = set(rpps["CODGEO"].dropna())
codes_finess = set(finess_commune["CODGEO"].dropna())

print("Communes INSEE :", len(codes_insee))
print("Communes RPPS :", len(codes_rpps))
print("Communes FINESS-SYGEN :", len(codes_finess))

print("\nCommunes INSEE sans RPPS :", len(codes_insee - codes_rpps))
print("Communes INSEE sans FINESS-SYGEN :", len(codes_insee - codes_finess))

print("\nCommunes RPPS hors INSEE :", len(codes_rpps - codes_insee))
print("Communes FINESS-SYGEN hors INSEE :", len(codes_finess - codes_insee))

Communes INSEE : 34858
Communes RPPS : 17213
Communes FINESS-SYGEN : 1757

Communes INSEE sans RPPS : 18143
Communes INSEE sans FINESS-SYGEN : 33148

Communes RPPS hors INSEE : 498
Communes FINESS-SYGEN hors INSEE : 47


In [ ]:
base_finale = insee.merge(
    rpps,
    on="CODGEO",
    how="left",
    validate="one_to_one"
)

base_finale["nb_professionnels"] = (
    base_finale["nb_professionnels"]
    .fillna(0)
    .astype(int)
)

print("Dimensions après fusion INSEE + RPPS :", base_finale.shape)

Dimensions après fusion INSEE + RPPS : (34858, 37)


In [ ]:
base_finale = base_finale.merge(
    finess_commune,
    on="CODGEO",
    how="left",
    validate="one_to_one"
)

base_finale["nb_etablissements"] = (
    base_finale["nb_etablissements"]
    .fillna(0)
    .astype(int)
)

print("Dimensions finales :", base_finale.shape)
print("Communes distinctes :", base_finale["CODGEO"].nunique())
print("Doublons CODGEO :", base_finale["CODGEO"].duplicated().sum())

assert len(base_finale) == len(insee)
assert not base_finale["CODGEO"].duplicated().any()

Dimensions finales : (34858, 106)
Communes distinctes : 34858
Doublons CODGEO : 0


In [ ]:
colonnes_controle = [
    "CODGEO",
    "LIBGEO",
    "nb_professionnels",
    "nb_etablissements",
    "nb_lits_mco",
    "nb_lits_smr",
    "nb_scanners",
    "nb_irm"
]

print("Dimensions :", base_finale.shape)

print("\nValeurs manquantes :")
print(base_finale[colonnes_controle].isna().sum())

base_finale.head()

Dimensions : (34858, 106)

Valeurs manquantes :
CODGEO                   0
LIBGEO                   0
nb_professionnels        0
nb_etablissements        0
nb_lits_mco          33950
nb_lits_smr          33587
nb_scanners          34114
nb_irm               34114
dtype: int64


,CODGEO,LIBGEO,SEXE1_AGEPYR1000,SEXE1_AGEPYR1003,SEXE1_AGEPYR1006,SEXE1_AGEPYR1011,SEXE1_AGEPYR1018,SEXE1_AGEPYR1025,SEXE1_AGEPYR1040,SEXE1_AGEPYR1055,...,nb_lits_usld,nb_places_had,effectif_salarie_total,effectif_liberal_total,etp_salarie_total,effectif_personnel_non_medical,etp_personnel_non_medical,nb_scanners,nb_irm,nb_mammographes
0,01001,L'Abergement-Clémenciat,19.000000,15.000000,36.000000,42.000000,15.000000,77.000000,87.000000,66.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01002,L'Abergement-de-Varey,3.000000,6.000000,7.000000,22.000000,6.000000,20.000000,39.000000,12.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01004,Ambérieu-en-Bugey,291.774098,327.193219,525.370992,719.637562,733.735172,1581.246385,1344.792069,833.964093,...,NaN,70.0,17.0,52.0,9.21,300.0,253.73,NaN,NaN,NaN
3,01005,Ambérieux-en-Dombes,36.262404,37.108922,65.943276,80.082047,54.445778,193.448597,196.440979,137.738813,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01006,Ambléon,2.000000,1.000000,2.000000,1.000000,4.000000,12.000000,11.000000,14.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
output_path = "s3://emma457/Projet_python/base_communes_finale.csv"

base_finale.to_csv(
    output_path,
    index=False
)

print("Table finale enregistrée :", output_path)

Table finale enregistrée : s3://emma457/Projet_python/base_communes_finale.csv
